In [2]:
# ============================================================
# SECTION 1 FIX — LOCATE TEST LABELS CORRECTLY
# ============================================================

from pathlib import Path

print("=" * 70)
print("LOCATING TEST LABELS")
print("=" * 70)

# Your images are:
# G:\EcoBotX_YOLO\images\test

TEST_IMAGES = Path(r"G:\EcoBotX_YOLO\images\test")

# Correct YOLO label location:
TEST_LABELS = Path(r"G:\EcoBotX_YOLO\labels\test")

print(f"\nTest images:")
print(f"  {TEST_IMAGES}")

print(f"\nTest labels:")
print(f"  {TEST_LABELS}")


# ============================================================
# CHECK DIRECTORIES
# ============================================================

if not TEST_IMAGES.exists():
    raise FileNotFoundError(
        f"[ERROR] Test images directory not found:\n{TEST_IMAGES}"
    )

if not TEST_LABELS.exists():
    raise FileNotFoundError(
        f"[ERROR] Test labels directory not found:\n{TEST_LABELS}"
    )


# ============================================================
# COUNT FILES
# ============================================================

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".webp"
}

test_image_files = sorted(
    [
        p for p in TEST_IMAGES.rglob("*")
        if p.is_file()
        and p.suffix.lower() in IMAGE_EXTENSIONS
    ]
)

test_label_files = sorted(
    TEST_LABELS.glob("*.txt")
)

print("\n" + "=" * 70)
print("DATASET CHECK")
print("=" * 70)

print(f"Test images : {len(test_image_files)}")
print(f"Test labels : {len(test_label_files)}")


# ============================================================
# VERIFY EXPECTED TEST SET
# ============================================================

assert len(test_image_files) == 1115, (
    f"Expected 1115 test images, found {len(test_image_files)}"
)

assert len(test_label_files) > 0, (
    "No test label files found."
)


print("\n[OK] Test images located successfully.")
print("[OK] Test labels located successfully.")


# ============================================================
# CHECK IMAGE/LABEL MATCHING
# ============================================================

image_stems = {p.stem for p in test_image_files}
label_stems = {p.stem for p in test_label_files}

missing_labels = image_stems - label_stems
extra_labels = label_stems - image_stems

print("\n" + "=" * 70)
print("IMAGE ↔ LABEL MATCHING")
print("=" * 70)

print(f"Images without labels : {len(missing_labels)}")
print(f"Labels without images : {len(extra_labels)}")

if missing_labels:
    print("\n[WARNING] Some images do not have labels.")
    print("Examples:")
    for name in sorted(missing_labels)[:10]:
        print(f"  {name}")

if extra_labels:
    print("\n[WARNING] Some labels do not have matching images.")
    print("Examples:")
    for name in sorted(extra_labels)[:10]:
        print(f"  {name}")


# ============================================================
# FINAL
# ============================================================

if not missing_labels and not extra_labels:
    print("\n[OK] Every test image has a matching label.")
else:
    print("\n[WARNING] Image/label mismatch detected.")

print("\n" + "=" * 70)
print("SECTION 1 LABEL LOCATION — COMPLETED")
print("=" * 70)

LOCATING TEST LABELS

Test images:
  G:\EcoBotX_YOLO\images\test

Test labels:
  G:\EcoBotX_YOLO\labels\test

DATASET CHECK
Test images : 1115
Test labels : 1115

[OK] Test images located successfully.
[OK] Test labels located successfully.

IMAGE ↔ LABEL MATCHING
Images without labels : 0
Labels without images : 0

[OK] Every test image has a matching label.

SECTION 1 LABEL LOCATION — COMPLETED


In [3]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 2 — GROUND-TRUTH CAN ANALYSIS
# ======================================================================

from pathlib import Path
from collections import Counter
import yaml
import numpy as np

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 2 — GROUND-TRUTH CAN ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. PATHS
# ======================================================================

DATASET_YAML = Path(r"G:\EcoBotX_YOLO\dataset.yaml")
TEST_IMAGES = Path(r"G:\EcoBotX_YOLO\images\test")
TEST_LABELS = Path(r"G:\EcoBotX_YOLO\labels\test")

CAN_CLASS_ID = 1

print("\n[1] Checking paths...")

assert DATASET_YAML.exists(), f"Dataset YAML not found: {DATASET_YAML}"
assert TEST_IMAGES.exists(), f"Test image directory not found: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Test label directory not found: {TEST_LABELS}"

print("[OK] Dataset YAML found")
print("[OK] Test images found")
print("[OK] Test labels found")


# ======================================================================
# 2. LOAD DATASET CONFIGURATION
# ======================================================================

with open(DATASET_YAML, "r", encoding="utf-8") as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg["names"]

if isinstance(names, dict):
    class_names = {int(k): v for k, v in names.items()}
else:
    class_names = {i: v for i, v in enumerate(names)}

print("\n" + "=" * 70)
print("DATASET CLASSES")
print("=" * 70)

for class_id, class_name in class_names.items():
    print(f"  {class_id}: {class_name}")


# ======================================================================
# 3. FIND TEST IMAGES
# ======================================================================

image_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

image_files = sorted(
    [
        p for p in TEST_IMAGES.iterdir()
        if p.is_file() and p.suffix.lower() in image_extensions
    ]
)

print("\n" + "=" * 70)
print("TEST DATASET")
print("=" * 70)

print(f"Test images: {len(image_files)}")
print(f"Test labels: {len(list(TEST_LABELS.glob('*.txt')))}")


# ======================================================================
# 4. READ CAN GROUND-TRUTH BOXES
# ======================================================================
#
# YOLO label format:
#
# class_id  x_center  y_center  width  height
#
# Coordinates are normalized to [0, 1].
#
# We only analyze:
#
#     class_id == 1  → CAN
#
# ======================================================================

can_annotations = []
can_images = []

total_annotations = 0
total_can_annotations = 0

class_counter = Counter()

missing_labels = []
invalid_labels = []

for image_path in image_files:

    label_path = TEST_LABELS / f"{image_path.stem}.txt"

    if not label_path.exists():
        missing_labels.append(image_path.name)
        continue

    image_has_can = False

    try:
        with open(label_path, "r", encoding="utf-8") as f:
            lines = [line.strip() for line in f if line.strip()]

        for line_number, line in enumerate(lines, start=1):

            parts = line.split()

            if len(parts) != 5:
                invalid_labels.append(
                    (image_path.name, line_number, line)
                )
                continue

            class_id = int(float(parts[0]))
            x_center = float(parts[1])
            y_center = float(parts[2])
            width = float(parts[3])
            height = float(parts[4])

            class_counter[class_id] += 1
            total_annotations += 1

            if class_id != CAN_CLASS_ID:
                continue

            # ----------------------------------------------------------
            # CAN annotation
            # ----------------------------------------------------------

            total_can_annotations += 1
            image_has_can = True

            area = width * height
            aspect_ratio = width / height if height > 0 else 0

            can_annotations.append({
                "image": image_path.name,
                "width": width,
                "height": height,
                "area": area,
                "aspect_ratio": aspect_ratio,
                "x_center": x_center,
                "y_center": y_center
            })

    except Exception as e:
        invalid_labels.append(
            (image_path.name, "ERROR", str(e))
        )

    if image_has_can:
        can_images.append(image_path.name)


# ======================================================================
# 5. BASIC CAN STATISTICS
# ======================================================================

print("\n" + "=" * 70)
print("CAN GROUND-TRUTH SUMMARY")
print("=" * 70)

print(f"Total test images              : {len(image_files)}")
print(f"Total valid annotations        : {total_annotations}")
print(f"Total CAN annotations          : {total_can_annotations}")
print(f"Images containing CAN          : {len(can_images)}")
print(f"Images without CAN             : {len(image_files) - len(can_images)}")

print("\nClass distribution in test labels:")

for class_id in sorted(class_counter):
    name = class_names.get(class_id, f"CLASS_{class_id}")
    print(
        f"  {name:<10}: {class_counter[class_id]}"
    )


# ======================================================================
# 6. CAN BOX SIZE STATISTICS
# ======================================================================

if total_can_annotations == 0:
    raise RuntimeError(
        "No CAN annotations were found. "
        "Check CAN_CLASS_ID and the label files."
    )

widths = np.array([x["width"] for x in can_annotations])
heights = np.array([x["height"] for x in can_annotations])
areas = np.array([x["area"] for x in can_annotations])
aspect_ratios = np.array(
    [x["aspect_ratio"] for x in can_annotations]
)


def print_stats(name, values):
    print(f"\n{name}")
    print(f"  Minimum : {np.min(values):.4f}")
    print(f"  Mean    : {np.mean(values):.4f}")
    print(f"  Median  : {np.median(values):.4f}")
    print(f"  Maximum : {np.max(values):.4f}")
    print(f"  Std     : {np.std(values):.4f}")


print("\n" + "=" * 70)
print("CAN BOUNDING-BOX STATISTICS")
print("=" * 70)

print_stats("Normalized Width", widths)
print_stats("Normalized Height", heights)
print_stats("Normalized Area", areas)
print_stats("Aspect Ratio (Width / Height)", aspect_ratios)


# ======================================================================
# 7. CAN SIZE CATEGORIES
# ======================================================================
#
# Area is normalized because YOLO coordinates are normalized.
#
# Small:
#     area < 0.02
#
# Medium:
#     0.02 <= area < 0.10
#
# Large:
#     area >= 0.10
#
# These categories are used only for diagnostic analysis.
# They are NOT COCO size definitions.
# ======================================================================

small_mask = areas < 0.02
medium_mask = (areas >= 0.02) & (areas < 0.10)
large_mask = areas >= 0.10

small_count = int(np.sum(small_mask))
medium_count = int(np.sum(medium_mask))
large_count = int(np.sum(large_mask))

print("\n" + "=" * 70)
print("CAN SIZE DISTRIBUTION")
print("=" * 70)

print(
    f"Small CANs  (< 2% image area)  : "
    f"{small_count} "
    f"({small_count / len(areas) * 100:.2f}%)"
)

print(
    f"Medium CANs (2–10% area)       : "
    f"{medium_count} "
    f"({medium_count / len(areas) * 100:.2f}%)"
)

print(
    f"Large CANs  (>= 10% area)      : "
    f"{large_count} "
    f"({large_count / len(areas) * 100:.2f}%)"
)


# ======================================================================
# 8. APPROXIMATE PIXEL SIZE AT 640x640
# ======================================================================
#
# Your YOLO experiments use imgsz=640.
#
# Since normalized YOLO width/height are relative to the image,
# we can estimate the corresponding bounding-box dimensions:
#
#     pixel_width  = normalized_width  * 640
#     pixel_height = normalized_height * 640
#
# This helps determine whether CAN objects are becoming very small
# after resizing.
# ======================================================================

pixel_widths = widths * 640
pixel_heights = heights * 640

print("\n" + "=" * 70)
print("APPROXIMATE CAN BOX SIZE AT 640x640")
print("=" * 70)

print_stats("Estimated Pixel Width", pixel_widths)
print_stats("Estimated Pixel Height", pixel_heights)


# ======================================================================
# 9. VERY SMALL CAN DETECTION
# ======================================================================
#
# Diagnostic threshold:
#
#     width < 32 pixels OR height < 32 pixels
#
# This does not mean the object is impossible to detect.
# It simply identifies potentially difficult small-object examples.
# ======================================================================

very_small_mask = (pixel_widths < 32) | (pixel_heights < 32)

very_small_count = int(np.sum(very_small_mask))

print("\n" + "=" * 70)
print("VERY SMALL CAN ANALYSIS")
print("=" * 70)

print(
    f"CAN boxes with width < 32 px OR height < 32 px: "
    f"{very_small_count}"
)

print(
    f"Percentage of CAN annotations: "
    f"{very_small_count / len(areas) * 100:.2f}%"
)


# ======================================================================
# 10. EXTREME ASPECT RATIOS
# ======================================================================

wide_mask = aspect_ratios > 2.0
tall_mask = aspect_ratios < 0.5

wide_count = int(np.sum(wide_mask))
tall_count = int(np.sum(tall_mask))

print("\n" + "=" * 70)
print("CAN SHAPE ANALYSIS")
print("=" * 70)

print(
    f"Very wide CAN boxes (ratio > 2.0): "
    f"{wide_count}"
)

print(
    f"Very tall CAN boxes (ratio < 0.5): "
    f"{tall_count}"
)


# ======================================================================
# 11. LOCATION ANALYSIS
# ======================================================================

x_centers = np.array([x["x_center"] for x in can_annotations])
y_centers = np.array([x["y_center"] for x in can_annotations])

print("\n" + "=" * 70)
print("CAN POSITION ANALYSIS")
print("=" * 70)

print(f"Mean X center : {np.mean(x_centers):.4f}")
print(f"Mean Y center : {np.mean(y_centers):.4f}")

print(f"Min X center  : {np.min(x_centers):.4f}")
print(f"Max X center  : {np.max(x_centers):.4f}")

print(f"Min Y center  : {np.min(y_centers):.4f}")
print(f"Max Y center  : {np.max(y_centers):.4f}")


# ======================================================================
# 12. TOP 20 SMALLEST CAN OBJECTS
# ======================================================================

sorted_can = sorted(
    can_annotations,
    key=lambda x: x["area"]
)

print("\n" + "=" * 70)
print("20 SMALLEST CAN ANNOTATIONS")
print("=" * 70)

for i, item in enumerate(sorted_can[:20], start=1):

    print(
        f"{i:02d}. "
        f"{item['image']:<35} "
        f"W={item['width']:.4f} "
        f"H={item['height']:.4f} "
        f"Area={item['area']:.5f} "
        f"AR={item['aspect_ratio']:.2f}"
    )


# ======================================================================
# 13. TOP 20 LARGEST CAN OBJECTS
# ======================================================================

print("\n" + "=" * 70)
print("20 LARGEST CAN ANNOTATIONS")
print("=" * 70)

for i, item in enumerate(sorted_can[-20:][::-1], start=1):

    print(
        f"{i:02d}. "
        f"{item['image']:<35} "
        f"W={item['width']:.4f} "
        f"H={item['height']:.4f} "
        f"Area={item['area']:.5f} "
        f"AR={item['aspect_ratio']:.2f}"
    )


# ======================================================================
# 14. FINAL SECTION 2 SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 2 — GROUND-TRUTH CAN ANALYSIS COMPLETED")
print("=" * 70)

print(f"""
CAN annotations       : {total_can_annotations}
Images containing CAN : {len(can_images)}

Mean CAN width        : {np.mean(widths):.4f}
Mean CAN height       : {np.mean(heights):.4f}
Mean CAN area         : {np.mean(areas):.5f}
Mean CAN aspect ratio : {np.mean(aspect_ratios):.3f}

Small CANs            : {small_count}
Medium CANs           : {medium_count}
Large CANs            : {large_count}

Very small CANs       : {very_small_count}
Very wide CANs        : {wide_count}
Very tall CANs        : {tall_count}
""")

print("[OK] SECTION 2 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 2 — GROUND-TRUTH CAN ANALYSIS

[1] Checking paths...
[OK] Dataset YAML found
[OK] Test images found
[OK] Test labels found

DATASET CLASSES
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

TEST DATASET
Test images: 1115
Test labels: 1115

CAN GROUND-TRUTH SUMMARY
Total test images              : 1115
Total valid annotations        : 899
Total CAN annotations          : 228
Images containing CAN          : 228
Images without CAN             : 887

Class distribution in test labels:
  BOTTLE    : 217
  CAN       : 228
  PAPER     : 250
  WRAPPER   : 204

CAN BOUNDING-BOX STATISTICS

Normalized Width
  Minimum : 0.1562
  Mean    : 0.5165
  Median  : 0.5000
  Maximum : 1.0000
  Std     : 0.2209

Normalized Height
  Minimum : 0.1484
  Mean    : 0.4770
  Median  : 0.4453
  Maximum : 1.0000
  Std     : 0.1966

Normalized Area
  Minimum : 0.0295
  Mean    : 0.2686
  Median  : 0.2028
  Maximum : 0.8914
  Std     : 0.1893

Aspect Ratio (Width / Height)
  

In [5]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS
# ======================================================================

from pathlib import Path
import yaml
import cv2
import numpy as np
from ultralytics import YOLO
from tqdm import tqdm
import shutil
import json

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt"
)

TEST_IMAGE_DIR = DATASET_ROOT / "images" / "test"
TEST_LABEL_DIR = DATASET_ROOT / "labels" / "test"

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
)

FAILURE_DIR = OUTPUT_DIR / "failures"

MISSED_DIR = FAILURE_DIR / "missed_can"
LOW_CONF_DIR = FAILURE_DIR / "low_confidence_can"
WRONG_CLASS_DIR = FAILURE_DIR / "wrong_class_can"
CORRECT_DIR = FAILURE_DIR / "correct_can"

for folder in [
    OUTPUT_DIR,
    MISSED_DIR,
    LOW_CONF_DIR,
    WRONG_CLASS_DIR,
    CORRECT_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)


# ======================================================================
# 2. PARAMETERS
# ======================================================================

CAN_CLASS_ID = 1

# IoU threshold for deciding whether prediction matches CAN ground truth
IOU_THRESHOLD = 0.50

# Confidence threshold used for normal detection
CONF_THRESHOLD = 0.25

# Additional threshold for identifying weak CAN detections
LOW_CONF_THRESHOLD = 0.50


# ======================================================================
# 3. CHECK PATHS
# ======================================================================

print("\n[1] Checking paths...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGE_DIR.exists(), f"Test image directory not found: {TEST_IMAGE_DIR}"
assert TEST_LABEL_DIR.exists(), f"Test label directory not found: {TEST_LABEL_DIR}"

print("[OK] Experiment 2 model found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")


# ======================================================================
# 4. LOAD MODEL
# ======================================================================

print("\n" + "=" * 70)
print("LOADING EXPERIMENT 2 MODEL")
print("=" * 70)

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded successfully")
print(f"Model: {MODEL_PATH}")

print("\nModel classes:")
for class_id, class_name in model.names.items():
    print(f"  {class_id}: {class_name}")


# ======================================================================
# 5. HELPER FUNCTIONS
# ======================================================================

def calculate_iou(box1, box2):
    """
    Calculate IoU between two boxes.

    Box format:
    [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersection_width = max(0, x2 - x1)
    intersection_height = max(0, y2 - y1)

    intersection_area = (
        intersection_width * intersection_height
    )

    area1 = max(0, box1[2] - box1[0]) * max(
        0, box1[3] - box1[1]
    )

    area2 = max(0, box2[2] - box2[0]) * max(
        0, box2[3] - box2[1]
    )

    union_area = area1 + area2 - intersection_area

    if union_area <= 0:
        return 0.0

    return intersection_area / union_area


def load_can_ground_truth(label_path, image_width, image_height):
    """
    Load CAN ground-truth boxes from YOLO label file.
    """

    can_boxes = []

    with open(label_path, "r") as f:
        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])

        if class_id != CAN_CLASS_ID:
            continue

        x_center = float(parts[1])
        y_center = float(parts[2])
        width = float(parts[3])
        height = float(parts[4])

        x1 = (x_center - width / 2) * image_width
        y1 = (y_center - height / 2) * image_height

        x2 = (x_center + width / 2) * image_width
        y2 = (y_center + height / 2) * image_height

        can_boxes.append(
            [x1, y1, x2, y2]
        )

    return can_boxes


# ======================================================================
# 6. ANALYSIS VARIABLES
# ======================================================================

total_can_gt = 0

true_positive_can = 0
missed_can = 0
low_conf_can = 0
wrong_class_can = 0

images_with_can = 0
images_with_missed_can = 0
images_with_wrong_class = 0
images_with_low_conf = 0

all_results = []

iou_values = []
confidence_values = []


# ======================================================================
# 7. GET TEST IMAGES
# ======================================================================

image_files = sorted(
    [
        p for p in TEST_IMAGE_DIR.iterdir()
        if p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp"
        ]
    ]
)

print("\n" + "=" * 70)
print("TEST DATASET")
print("=" * 70)

print(f"Test images: {len(image_files)}")


# ======================================================================
# 8. RUN EXPERIMENT 2
# ======================================================================

print("\n" + "=" * 70)
print("RUNNING EXPERIMENT 2 ON TEST SET")
print("=" * 70)

for image_path in tqdm(
    image_files,
    desc="Analyzing test images"
):

    label_path = TEST_LABEL_DIR / f"{image_path.stem}.txt"

    # --------------------------------------------------------------
    # Read image
    # --------------------------------------------------------------

    image = cv2.imread(str(image_path))

    if image is None:
        continue

    image_height, image_width = image.shape[:2]

    # --------------------------------------------------------------
    # Load CAN ground truth
    # --------------------------------------------------------------

    gt_can_boxes = load_can_ground_truth(
        label_path,
        image_width,
        image_height
    )

    if len(gt_can_boxes) == 0:
        continue

    images_with_can += 1
    total_can_gt += len(gt_can_boxes)

    # --------------------------------------------------------------
    # Run prediction
    # --------------------------------------------------------------

    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        iou=0.7,
        verbose=False
    )

    result = results[0]

    predictions = []

    if result.boxes is not None:

        for box in result.boxes:

            xyxy = box.xyxy[0].cpu().numpy()

            cls = int(
                box.cls[0].cpu().item()
            )

            conf = float(
                box.conf[0].cpu().item()
            )

            predictions.append(
                {
                    "box": xyxy.tolist(),
                    "class_id": cls,
                    "confidence": conf
                }
            )

    # --------------------------------------------------------------
    # Match every CAN ground truth
    # --------------------------------------------------------------

    image_status = []

    for gt_index, gt_box in enumerate(gt_can_boxes):

        best_iou = 0.0
        best_prediction = None

        # Find prediction with highest IoU
        for pred in predictions:

            iou = calculate_iou(
                gt_box,
                pred["box"]
            )

            if iou > best_iou:
                best_iou = iou
                best_prediction = pred

        # ==========================================================
        # CASE 1 — CORRECT CAN DETECTION
        # ==========================================================

        if (
            best_prediction is not None
            and best_iou >= IOU_THRESHOLD
            and best_prediction["class_id"] == CAN_CLASS_ID
        ):

            true_positive_can += 1

            confidence_values.append(
                best_prediction["confidence"]
            )

            iou_values.append(best_iou)

            image_status.append("CORRECT")

            # Save only if confidence is relatively weak
            if best_prediction["confidence"] < LOW_CONF_THRESHOLD:

                low_conf_can += 1
                images_with_low_conf += 1

                shutil.copy2(
                    image_path,
                    LOW_CONF_DIR / image_path.name
                )

        # ==========================================================
        # CASE 2 — WRONG CLASS
        # ==========================================================

        elif (
            best_prediction is not None
            and best_iou >= IOU_THRESHOLD
            and best_prediction["class_id"] != CAN_CLASS_ID
        ):

            wrong_class_can += 1
            images_with_wrong_class += 1

            predicted_class = model.names[
                best_prediction["class_id"]
            ]

            image_status.append(
                f"WRONG_CLASS_{predicted_class}"
            )

            # Draw analysis image
            annotated = image.copy()

            # Ground truth CAN = green
            x1, y1, x2, y2 = map(
                int,
                gt_box
            )

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

            # Prediction = red
            px1, py1, px2, py2 = map(
                int,
                best_prediction["box"]
            )

            cv2.rectangle(
                annotated,
                (px1, py1),
                (px2, py2),
                (0, 0, 255),
                3
            )

            label = (
                f"GT: CAN | "
                f"Pred: {predicted_class} | "
                f"Conf: {best_prediction['confidence']:.2f} | "
                f"IoU: {best_iou:.2f}"
            )

            cv2.putText(
                annotated,
                label,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 0, 255),
                2
            )

            cv2.imwrite(
                str(
                    WRONG_CLASS_DIR /
                    image_path.name
                ),
                annotated
            )

        # ==========================================================
        # CASE 3 — MISSED CAN
        # ==========================================================

        else:

            missed_can += 1
            images_with_missed_can += 1

            image_status.append("MISSED")

            # Draw ground-truth box
            annotated = image.copy()

            x1, y1, x2, y2 = map(
                int,
                gt_box
            )

            cv2.rectangle(
                annotated,
                (x1, y1),
                (x2, y2),
                (0, 255, 0),
                3
            )

            label = (
                f"CAN MISSED | "
                f"Best IoU: {best_iou:.2f}"
            )

            cv2.putText(
                annotated,
                label,
                (20, 35),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 0, 255),
                2
            )

            cv2.imwrite(
                str(
                    MISSED_DIR /
                    image_path.name
                ),
                annotated
            )

    # --------------------------------------------------------------
    # Save image-level result
    # --------------------------------------------------------------

    all_results.append(
        {
            "image": image_path.name,
            "ground_truth_can": len(gt_can_boxes),
            "status": image_status
        }
    )


# ======================================================================
# 9. FINAL STATISTICS
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 3 — CAN PREDICTION RESULTS")
print("=" * 70)

print(f"\nTotal CAN ground-truth boxes : {total_can_gt}")

print(f"Correct CAN detections       : {true_positive_can}")
print(f"Missed CAN boxes             : {missed_can}")
print(f"Wrong-class CAN boxes        : {wrong_class_can}")
print(f"Low-confidence CAN detections: {low_conf_can}")


# ======================================================================
# 10. CALCULATE CAN RECALL
# ======================================================================

if total_can_gt > 0:

    can_recall = (
        true_positive_can /
        total_can_gt
    )

else:

    can_recall = 0.0


print("\n" + "=" * 70)
print("CAN-SPECIFIC PERFORMANCE")
print("=" * 70)

print(
    f"CAN Recall: "
    f"{can_recall:.4f} "
    f"({can_recall * 100:.2f}%)"
)


# ======================================================================
# 11. IOU STATISTICS
# ======================================================================

if iou_values:

    print("\n" + "=" * 70)
    print("CAN LOCALIZATION QUALITY")
    print("=" * 70)

    print(
        f"Mean IoU   : {np.mean(iou_values):.4f}"
    )

    print(
        f"Median IoU : {np.median(iou_values):.4f}"
    )

    print(
        f"Minimum IoU: {np.min(iou_values):.4f}"
    )

    print(
        f"Maximum IoU: {np.max(iou_values):.4f}"
    )


# ======================================================================
# 12. CONFIDENCE STATISTICS
# ======================================================================

if confidence_values:

    print("\n" + "=" * 70)
    print("CAN CONFIDENCE ANALYSIS")
    print("=" * 70)

    print(
        f"Mean confidence   : "
        f"{np.mean(confidence_values):.4f}"
    )

    print(
        f"Median confidence : "
        f"{np.median(confidence_values):.4f}"
    )

    print(
        f"Minimum confidence: "
        f"{np.min(confidence_values):.4f}"
    )

    print(
        f"Maximum confidence: "
        f"{np.max(confidence_values):.4f}"
    )


# ======================================================================
# 13. SAVE JSON RESULTS
# ======================================================================

json_path = OUTPUT_DIR / "can_prediction_analysis.json"

summary = {
    "experiment": "Experiment 2",
    "model": str(MODEL_PATH),
    "test_images": len(image_files),
    "can_ground_truth": total_can_gt,
    "correct_can": true_positive_can,
    "missed_can": missed_can,
    "wrong_class_can": wrong_class_can,
    "low_confidence_can": low_conf_can,
    "can_recall": can_recall,
    "iou_mean": float(np.mean(iou_values))
        if iou_values else None,
    "iou_median": float(np.median(iou_values))
        if iou_values else None,
    "confidence_mean": float(np.mean(confidence_values))
        if confidence_values else None,
    "confidence_median": float(np.median(confidence_values))
        if confidence_values else None,
}

with open(
    json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        {
            "summary": summary,
            "images": all_results
        },
        f,
        indent=4
    )


# ======================================================================
# 14. OUTPUT DIRECTORIES
# ======================================================================

print("\n" + "=" * 70)
print("FAILURE IMAGE OUTPUT")
print("=" * 70)

print(
    f"Missed CAN images       : "
    f"{MISSED_DIR}"
)

print(
    f"Wrong-class CAN images  : "
    f"{WRONG_CLASS_DIR}"
)

print(
    f"Low-confidence CAN      : "
    f"{LOW_CONF_DIR}"
)

print(
    f"Analysis JSON            : "
    f"{json_path}"
)


# ======================================================================
# 15. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 3 — COMPLETED")
print("=" * 70)

print(
    f"CAN Ground Truth : {total_can_gt}"
)

print(
    f"Correct          : {true_positive_can}"
)

print(
    f"Missed           : {missed_can}"
)

print(
    f"Wrong Class      : {wrong_class_can}"
)

print(
    f"Low Confidence   : {low_conf_can}"
)

print(
    f"CAN Recall       : {can_recall * 100:.2f}%"
)

print("\n[OK] SECTION 3 COMPLETED")


EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 3 — EXPERIMENT 2 CAN PREDICTION ANALYSIS

[1] Checking paths...
[OK] Experiment 2 model found
[OK] Test image directory found
[OK] Test label directory found

LOADING EXPERIMENT 2 MODEL
[OK] Model loaded successfully
Model: G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt

Model classes:
  0: BOTTLE
  1: CAN
  2: PAPER
  3: WRAPPER

TEST DATASET
Test images: 1115

RUNNING EXPERIMENT 2 ON TEST SET


Analyzing test images: 100%|██████████| 1115/1115 [00:04<00:00, 231.75it/s]


SECTION 3 — CAN PREDICTION RESULTS

Total CAN ground-truth boxes : 228
Correct CAN detections       : 196
Missed CAN boxes             : 22
Wrong-class CAN boxes        : 10
Low-confidence CAN detections: 16

CAN-SPECIFIC PERFORMANCE
CAN Recall: 0.8596 (85.96%)

CAN LOCALIZATION QUALITY
Mean IoU   : 0.8311
Median IoU : 0.8458
Minimum IoU: 0.5119
Maximum IoU: 0.9884

CAN CONFIDENCE ANALYSIS
Mean confidence   : 0.7575
Median confidence : 0.8032
Minimum confidence: 0.2517
Maximum confidence: 0.9498

FAILURE IMAGE OUTPUT
Missed CAN images       : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\missed_can
Wrong-class CAN images  : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\wrong_class_can
Low-confidence CAN      : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\failures\low_confidence_can
Analysis JSON            : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\can_prediction_analysis.json

SECTION 3 — COMPLETED
CAN Ground Tru

In [6]:

# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 4 — CAN FAILURE CHARACTERIZATION
# ======================================================================

from pathlib import Path
import cv2
import numpy as np
import json
from collections import Counter, defaultdict

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 4 — CAN FAILURE CHARACTERIZATION")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

TEST_IMAGE_DIR = DATASET_ROOT / "images" / "test"
TEST_LABEL_DIR = DATASET_ROOT / "labels" / "test"

SECTION3_JSON = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\can_prediction_analysis.json"
)

OUTPUT_DIR = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
)

SECTION4_JSON = OUTPUT_DIR / "can_failure_characterization.json"

CAN_CLASS_ID = 1

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}


# ======================================================================
# 2. CHECK FILES
# ======================================================================

print("\n[1] Checking files...")

assert TEST_IMAGE_DIR.exists(), (
    f"Test image directory not found:\n{TEST_IMAGE_DIR}"
)

assert TEST_LABEL_DIR.exists(), (
    f"Test label directory not found:\n{TEST_LABEL_DIR}"
)

assert SECTION3_JSON.exists(), (
    f"Section 3 JSON not found:\n{SECTION3_JSON}"
)

print("[OK] Test image directory found")
print("[OK] Test label directory found")
print("[OK] Section 3 results found")


# ======================================================================
# 3. LOAD SECTION 3 RESULTS
# ======================================================================

print("\n[2] Loading Section 3 results...")

with open(
    SECTION3_JSON,
    "r",
    encoding="utf-8"
) as f:

    section3_data = json.load(f)

print("[OK] Section 3 results loaded")


# ======================================================================
# 4. BUILD FAILURE LOOKUP
# ======================================================================

print("\n[3] Building failure lookup...")

image_results = section3_data["images"]

failure_lookup = {}

for item in image_results:

    image_name = item["image"]

    statuses = item.get("status", [])

    failure_lookup[image_name] = statuses

print(
    f"[OK] Results loaded for "
    f"{len(failure_lookup)} images"
)


# ======================================================================
# 5. HELPER FUNCTIONS
# ======================================================================

def read_can_boxes(label_path):

    can_boxes = []

    with open(
        label_path,
        "r",
        encoding="utf-8"
    ) as f:

        lines = f.readlines()

    for line in lines:

        parts = line.strip().split()

        if len(parts) != 5:
            continue

        class_id = int(parts[0])

        if class_id != CAN_CLASS_ID:
            continue

        xc = float(parts[1])
        yc = float(parts[2])
        w = float(parts[3])
        h = float(parts[4])

        area = w * h

        aspect_ratio = (
            w / h
            if h > 0
            else 0
        )

        can_boxes.append(
            {
                "xc": xc,
                "yc": yc,
                "width": w,
                "height": h,
                "area": area,
                "aspect_ratio": aspect_ratio
            }
        )

    return can_boxes


def classify_size(area):

    if area < 0.02:
        return "SMALL"

    elif area < 0.10:
        return "MEDIUM"

    else:
        return "LARGE"


def classify_shape(aspect_ratio):

    if aspect_ratio > 2.0:
        return "VERY_WIDE"

    elif aspect_ratio < 0.5:
        return "VERY_TALL"

    else:
        return "NORMAL"


def classify_position(xc, yc):

    # Horizontal position
    if xc < 0.33:
        horizontal = "LEFT"

    elif xc > 0.67:
        horizontal = "RIGHT"

    else:
        horizontal = "CENTER"

    # Vertical position
    if yc < 0.33:
        vertical = "TOP"

    elif yc > 0.67:
        vertical = "BOTTOM"

    else:
        vertical = "CENTER"

    return horizontal, vertical


# ======================================================================
# 6. COLLECT CAN FAILURE DATA
# ======================================================================

print("\n" + "=" * 70)
print("ANALYZING CAN FAILURE CHARACTERISTICS")
print("=" * 70)


all_can = []
missed_can = []
wrong_class_can = []
correct_can = []

wrong_class_counter = Counter()


# ======================================================================
# 7. PROCESS EVERY TEST IMAGE
# ======================================================================

image_files = sorted(
    [
        p for p in TEST_IMAGE_DIR.iterdir()
        if p.suffix.lower() in [
            ".jpg",
            ".jpeg",
            ".png",
            ".bmp"
        ]
    ]
)


for image_path in image_files:

    image_name = image_path.name

    if image_name not in failure_lookup:
        continue

    label_path = TEST_LABEL_DIR / (
        image_path.stem + ".txt"
    )

    if not label_path.exists():
        continue

    can_boxes = read_can_boxes(label_path)

    if not can_boxes:
        continue

    statuses = failure_lookup[image_name]

    # --------------------------------------------------------------
    # Analyze each CAN annotation
    # --------------------------------------------------------------

    for index, can in enumerate(can_boxes):

        size_category = classify_size(
            can["area"]
        )

        shape_category = classify_shape(
            can["aspect_ratio"]
        )

        horizontal, vertical = classify_position(
            can["xc"],
            can["yc"]
        )

        record = {
            "image": image_name,
            "width": can["width"],
            "height": can["height"],
            "area": can["area"],
            "aspect_ratio": can["aspect_ratio"],
            "xc": can["xc"],
            "yc": can["yc"],
            "size": size_category,
            "shape": shape_category,
            "horizontal": horizontal,
            "vertical": vertical
        }

        all_can.append(record)

        # ----------------------------------------------------------
        # Determine status
        # ----------------------------------------------------------

        status = (
            statuses[index]
            if index < len(statuses)
            else "UNKNOWN"
        )

        if status == "CORRECT":
            correct_can.append(record)

        elif status == "MISSED":
            missed_can.append(record)

        elif status.startswith("WRONG_CLASS_"):

            wrong_class_can.append(record)

            predicted_class = (
                status.replace(
                    "WRONG_CLASS_",
                    ""
                )
            )

            record["predicted_class"] = (
                predicted_class
            )

            wrong_class_counter[
                predicted_class
            ] += 1


# ======================================================================
# 8. BASIC FAILURE COUNTS
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE COUNTS")
print("=" * 70)

print(
    f"Total CAN annotations : "
    f"{len(all_can)}"
)

print(
    f"Correct CAN           : "
    f"{len(correct_can)}"
)

print(
    f"Missed CAN            : "
    f"{len(missed_can)}"
)

print(
    f"Wrong-class CAN       : "
    f"{len(wrong_class_can)}"
)


# ======================================================================
# 9. FAILURE RATE
# ======================================================================

total_failures = (
    len(missed_can)
    + len(wrong_class_can)
)

if len(all_can) > 0:

    failure_rate = (
        total_failures /
        len(all_can)
    )

else:

    failure_rate = 0.0


print(
    f"\nOverall CAN failure rate : "
    f"{failure_rate * 100:.2f}%"
)


# ======================================================================
# 10. SIZE ANALYSIS
# ======================================================================

def distribution(records, key):

    counter = Counter(
        r[key]
        for r in records
    )

    return dict(counter)


print("\n" + "=" * 70)
print("CAN FAILURE BY OBJECT SIZE")
print("=" * 70)

size_all = distribution(
    all_can,
    "size"
)

size_missed = distribution(
    missed_can,
    "size"
)

size_wrong = distribution(
    wrong_class_can,
    "size"
)

for category in [
    "SMALL",
    "MEDIUM",
    "LARGE"
]:

    total = size_all.get(
        category,
        0
    )

    missed = size_missed.get(
        category,
        0
    )

    wrong = size_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 11. ASPECT-RATIO ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY SHAPE")
print("=" * 70)

shape_all = distribution(
    all_can,
    "shape"
)

shape_missed = distribution(
    missed_can,
    "shape"
)

shape_wrong = distribution(
    wrong_class_can,
    "shape"
)

for category in [
    "VERY_WIDE",
    "VERY_TALL",
    "NORMAL"
]:

    total = shape_all.get(
        category,
        0
    )

    missed = shape_missed.get(
        category,
        0
    )

    wrong = shape_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<11} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 12. POSITION ANALYSIS — HORIZONTAL
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY HORIZONTAL POSITION")
print("=" * 70)

horizontal_all = distribution(
    all_can,
    "horizontal"
)

horizontal_missed = distribution(
    missed_can,
    "horizontal"
)

horizontal_wrong = distribution(
    wrong_class_can,
    "horizontal"
)

for category in [
    "LEFT",
    "CENTER",
    "RIGHT"
]:

    total = horizontal_all.get(
        category,
        0
    )

    missed = horizontal_missed.get(
        category,
        0
    )

    wrong = horizontal_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 13. POSITION ANALYSIS — VERTICAL
# ======================================================================

print("\n" + "=" * 70)
print("CAN FAILURE BY VERTICAL POSITION")
print("=" * 70)

vertical_all = distribution(
    all_can,
    "vertical"
)

vertical_missed = distribution(
    missed_can,
    "vertical"
)

vertical_wrong = distribution(
    wrong_class_can,
    "vertical"
)

for category in [
    "TOP",
    "CENTER",
    "BOTTOM"
]:

    total = vertical_all.get(
        category,
        0
    )

    missed = vertical_missed.get(
        category,
        0
    )

    wrong = vertical_wrong.get(
        category,
        0
    )

    failures = missed + wrong

    rate = (
        failures / total * 100
        if total > 0
        else 0
    )

    print(
        f"{category:<8} "
        f"Total={total:<4} "
        f"Missed={missed:<4} "
        f"Wrong={wrong:<4} "
        f"Failure Rate={rate:.2f}%"
    )


# ======================================================================
# 14. NUMERICAL STATISTICS
# ======================================================================

def print_numeric_stats(
    name,
    records,
    key
):

    values = [
        r[key]
        for r in records
        if key in r
    ]

    if not values:
        return

    print(
        f"\n{name}"
    )

    print(
        f"  Mean   : {np.mean(values):.4f}"
    )

    print(
        f"  Median : {np.median(values):.4f}"
    )

    print(
        f"  Min    : {np.min(values):.4f}"
    )

    print(
        f"  Max    : {np.max(values):.4f}"
    )


print("\n" + "=" * 70)
print("NUMERICAL FAILURE CHARACTERISTICS")
print("=" * 70)

print_numeric_stats(
    "ALL CAN",
    all_can,
    "area"
)

print_numeric_stats(
    "MISSED CAN",
    missed_can,
    "area"
)

print_numeric_stats(
    "WRONG-CLASS CAN",
    wrong_class_can,
    "area"
)

print_numeric_stats(
    "ALL CAN ASPECT RATIO",
    all_can,
    "aspect_ratio"
)

print_numeric_stats(
    "MISSED CAN ASPECT RATIO",
    missed_can,
    "aspect_ratio"
)

print_numeric_stats(
    "WRONG-CLASS CAN ASPECT RATIO",
    wrong_class_can,
    "aspect_ratio"
)


# ======================================================================
# 15. WRONG CLASS CONFUSION ANALYSIS
# ======================================================================

print("\n" + "=" * 70)
print("CAN WRONG-CLASS CONFUSION")
print("=" * 70)

if wrong_class_counter:

    for class_name, count in (
        wrong_class_counter.most_common()
    ):

        percentage = (
            count /
            len(wrong_class_can) *
            100
        )

        print(
            f"CAN → {class_name:<8} "
            f"{count:>3} "
            f"({percentage:.2f}%)"
        )

else:

    print("No wrong-class CAN cases found.")


# ======================================================================
# 16. FAILURE CASES WITH THEIR PROPERTIES
# ======================================================================

print("\n" + "=" * 70)
print("MISSED CAN CASES")
print("=" * 70)

for i, record in enumerate(
    missed_can,
    start=1
):

    print(
        f"{i:02d}. "
        f"{record['image']} | "
        f"Area={record['area']:.4f} | "
        f"AR={record['aspect_ratio']:.2f} | "
        f"Center=({record['xc']:.2f}, "
        f"{record['yc']:.2f}) | "
        f"Size={record['size']} | "
        f"Shape={record['shape']}"
    )


# ======================================================================
# 17. WRONG CLASS CASES
# ======================================================================

print("\n" + "=" * 70)
print("WRONG-CLASS CAN CASES")
print("=" * 70)

for i, record in enumerate(
    wrong_class_can,
    start=1
):

    print(
        f"{i:02d}. "
        f"{record['image']} | "
        f"Predicted={record.get('predicted_class', 'UNKNOWN')} | "
        f"Area={record['area']:.4f} | "
        f"AR={record['aspect_ratio']:.2f} | "
        f"Center=({record['xc']:.2f}, "
        f"{record['yc']:.2f}) | "
        f"Size={record['size']} | "
        f"Shape={record['shape']}"
    )


# ======================================================================
# 18. SAVE RESULTS
# ======================================================================

output_data = {

    "total_can": len(all_can),

    "correct_can": len(correct_can),

    "missed_can": len(missed_can),

    "wrong_class_can": len(wrong_class_can),

    "failure_rate": failure_rate,

    "size": {
        "all": size_all,
        "missed": size_missed,
        "wrong_class": size_wrong
    },

    "shape": {
        "all": shape_all,
        "missed": shape_missed,
        "wrong_class": shape_wrong
    },

    "horizontal": {
        "all": horizontal_all,
        "missed": horizontal_missed,
        "wrong_class": horizontal_wrong
    },

    "vertical": {
        "all": vertical_all,
        "missed": vertical_missed,
        "wrong_class": vertical_wrong
    },

    "wrong_class_confusion": dict(
        wrong_class_counter
    ),

    "missed_cases": missed_can,

    "wrong_class_cases": wrong_class_can
}


with open(
    SECTION4_JSON,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        output_data,
        f,
        indent=4
    )


# ======================================================================
# 19. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 4 — FINAL SUMMARY")
print("=" * 70)

print(
    f"Total CAN          : {len(all_can)}"
)

print(
    f"Correct            : {len(correct_can)}"
)

print(
    f"Missed             : {len(missed_can)}"
)

print(
    f"Wrong class        : {len(wrong_class_can)}"
)

print(
    f"Total failures     : {total_failures}"
)

print(
    f"Failure rate       : "
    f"{failure_rate * 100:.2f}%"
)

print("\nWrong-class confusion:")

if wrong_class_counter:

    for cls, count in wrong_class_counter.items():

        print(
            f"  CAN → {cls}: {count}"
        )

else:

    print("  None")


print("\nSaved characterization results:")
print(SECTION4_JSON)

print("\n" + "=" * 70)
print("[OK] SECTION 4 COMPLETED")
print("=" * 70)



EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 4 — CAN FAILURE CHARACTERIZATION

[1] Checking files...
[OK] Test image directory found
[OK] Test label directory found
[OK] Section 3 results found

[2] Loading Section 3 results...
[OK] Section 3 results loaded

[3] Building failure lookup...
[OK] Results loaded for 228 images

ANALYZING CAN FAILURE CHARACTERISTICS

CAN FAILURE COUNTS
Total CAN annotations : 228
Correct CAN           : 196
Missed CAN            : 22
Wrong-class CAN       : 10

Overall CAN failure rate : 14.04%

CAN FAILURE BY OBJECT SIZE
SMALL    Total=0    Missed=0    Wrong=0    Failure Rate=0.00%
MEDIUM   Total=44   Missed=3    Wrong=0    Failure Rate=6.82%
LARGE    Total=184  Missed=19   Wrong=10   Failure Rate=15.76%

CAN FAILURE BY SHAPE
VERY_WIDE   Total=21   Missed=1    Wrong=1    Failure Rate=9.52%
VERY_TALL   Total=13   Missed=1    Wrong=0    Failure Rate=7.69%
NORMAL      Total=194  Missed=20   Wrong=9    Failure Rate=14.95%

CAN FAILURE BY HORIZONTAL POSITION
LEF

In [7]:
# ======================================================================
# EXPERIMENT 2 — CAN FAILURE ANALYSIS
# SECTION 5 — VISUAL CAN FAILURE INSPECTION
# ======================================================================

from pathlib import Path
import json
import math
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("=" * 70)
print("EXPERIMENT 2 — CAN FAILURE ANALYSIS")
print("SECTION 5 — VISUAL CAN FAILURE INSPECTION")
print("=" * 70)


# ======================================================================
# 1. CONFIGURATION
# ======================================================================

DATASET_ROOT = Path(r"G:\EcoBotX_YOLO")

TEST_IMAGES = DATASET_ROOT / "images" / "test"
TEST_LABELS = DATASET_ROOT / "labels" / "test"

SECTION4_JSON = Path(
    r"G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis"
    r"\can_failure_characterization.json"
)

MODEL_PATH = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\experiment2_ecobotx_light-2\weights\best.pt"
)

OUTPUT_ROOT = Path(
    r"G:\EcoBotX_YOLO_training"
    r"\experiment2_can_failure_analysis"
    r"\section5_visual_inspection"
)

MISSED_DIR = OUTPUT_ROOT / "missed_can"
WRONG_DIR = OUTPUT_ROOT / "wrong_class_can"

MISSED_DIR.mkdir(parents=True, exist_ok=True)
WRONG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = {
    0: "BOTTLE",
    1: "CAN",
    2: "PAPER",
    3: "WRAPPER"
}

CAN_ID = 1

IOU_THRESHOLD = 0.50
CONF_THRESHOLD = 0.25

print("\n[1] Configuration")
print("-" * 70)

print(f"Model       : {MODEL_PATH}")
print(f"Test images : {TEST_IMAGES}")
print(f"Test labels : {TEST_LABELS}")
print(f"Section 4   : {SECTION4_JSON}")
print(f"Output      : {OUTPUT_ROOT}")


# ======================================================================
# 2. CHECK FILES
# ======================================================================

print("\n[2] Checking required files...")

assert MODEL_PATH.exists(), f"Model not found: {MODEL_PATH}"
assert TEST_IMAGES.exists(), f"Test image directory not found: {TEST_IMAGES}"
assert TEST_LABELS.exists(), f"Test label directory not found: {TEST_LABELS}"
assert SECTION4_JSON.exists(), f"Section 4 JSON not found: {SECTION4_JSON}"

print("[OK] Model found")
print("[OK] Test image directory found")
print("[OK] Test label directory found")
print("[OK] Section 4 characterization found")


# ======================================================================
# 3. LOAD MODEL
# ======================================================================

print("\n[3] Loading Experiment 2 model...")

from ultralytics import YOLO

model = YOLO(str(MODEL_PATH))

print("[OK] Model loaded")
print(f"Model classes: {model.names}")


# ======================================================================
# 4. LOAD SECTION 4 RESULTS
# ======================================================================

print("\n[4] Loading Section 4 results...")

with open(SECTION4_JSON, "r", encoding="utf-8") as f:
    characterization = json.load(f)

print("[OK] Section 4 results loaded")


# ======================================================================
# 5. EXTRACT FAILURE FILENAMES
# ======================================================================

print("\n[5] Extracting failure cases...")

missed_cases = []
wrong_cases = []

# Section 4 JSON structure can vary slightly depending on previous code.
# We search recursively for dictionaries containing filename/failure data.

def recursive_collect(obj):
    if isinstance(obj, dict):

        # Look for missed CAN information
        failure_type = str(
            obj.get("failure_type", obj.get("type", ""))
        ).lower()

        filename = (
            obj.get("filename")
            or obj.get("image")
            or obj.get("image_name")
            or obj.get("file")
        )

        predicted = (
            obj.get("predicted_class")
            or obj.get("predicted")
            or obj.get("prediction")
        )

        if filename:

            record = {
                "filename": str(filename),
                "predicted": predicted,
                "failure_type": failure_type
            }

            if "miss" in failure_type:
                missed_cases.append(record)

            elif "wrong" in failure_type:
                wrong_cases.append(record)

        for value in obj.values():
            recursive_collect(value)

    elif isinstance(obj, list):
        for item in obj:
            recursive_collect(item)


recursive_collect(characterization)

# Remove duplicates
def unique_cases(cases):
    seen = set()
    result = []

    for c in cases:
        key = c["filename"]

        if key not in seen:
            seen.add(key)
            result.append(c)

    return result


missed_cases = unique_cases(missed_cases)
wrong_cases = unique_cases(wrong_cases)

print(f"[INFO] Missed cases found      : {len(missed_cases)}")
print(f"[INFO] Wrong-class cases found : {len(wrong_cases)}")


# ======================================================================
# 6. FALLBACK — READ FAILURE DIRECTORIES
# ======================================================================

# If Section 4 JSON does not contain filenames in a machine-readable
# structure, use the folders produced during Section 3.

if len(missed_cases) == 0:

    print("\n[INFO] Searching missed CAN directory...")

    section3_missed = Path(
        r"G:\EcoBotX_YOLO_training"
        r"\experiment2_can_failure_analysis"
        r"\failures\missed_can"
    )

    if section3_missed.exists():

        for p in section3_missed.iterdir():

            if p.is_file():

                missed_cases.append({
                    "filename": p.name,
                    "predicted": None,
                    "failure_type": "missed"
                })

if len(wrong_cases) == 0:

    print("[INFO] Searching wrong-class CAN directory...")

    section3_wrong = Path(
        r"G:\EcoBotX_YOLO_training"
        r"\experiment2_can_failure_analysis"
        r"\failures\wrong_class_can"
    )

    if section3_wrong.exists():

        for p in section3_wrong.iterdir():

            if p.is_file():

                wrong_cases.append({
                    "filename": p.name,
                    "predicted": None,
                    "failure_type": "wrong_class"
                })


print("\nFailure cases available:")
print(f"  Missed CAN      : {len(missed_cases)}")
print(f"  Wrong-class CAN : {len(wrong_cases)}")


# ======================================================================
# 7. HELPER FUNCTIONS
# ======================================================================

def load_yolo_labels(label_path, image_width, image_height):

    boxes = []

    if not label_path.exists():
        return boxes

    with open(label_path, "r", encoding="utf-8") as f:

        for line in f:

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            cls = int(parts[0])
            xc = float(parts[1])
            yc = float(parts[2])
            w = float(parts[3])
            h = float(parts[4])

            x1 = (xc - w / 2) * image_width
            y1 = (yc - h / 2) * image_height
            x2 = (xc + w / 2) * image_width
            y2 = (yc + h / 2) * image_height

            boxes.append({
                "class_id": cls,
                "class_name": CLASS_NAMES.get(cls, str(cls)),
                "xyxy": [x1, y1, x2, y2],
                "xc": xc,
                "yc": yc,
                "w": w,
                "h": h,
                "area": w * h,
                "aspect_ratio": w / h if h > 0 else 0
            })

    return boxes


def calculate_iou(box_a, box_b):

    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b

    ix1 = max(ax1, bx1)
    iy1 = max(ay1, by1)
    ix2 = min(ax2, bx2)
    iy2 = min(ay2, by2)

    iw = max(0, ix2 - ix1)
    ih = max(0, iy2 - iy1)

    intersection = iw * ih

    area_a = max(0, ax2 - ax1) * max(0, ay2 - ay1)
    area_b = max(0, bx2 - bx1) * max(0, by2 - by1)

    union = area_a + area_b - intersection

    if union <= 0:
        return 0.0

    return intersection / union


def find_image(filename):

    exact = TEST_IMAGES / filename

    if exact.exists():
        return exact

    matches = list(TEST_IMAGES.glob(filename))

    if matches:
        return matches[0]

    # Try matching by filename suffix
    for p in TEST_IMAGES.iterdir():

        if p.name == filename:
            return p

    return None


def find_label(image_path):

    return TEST_LABELS / f"{image_path.stem}.txt"


# ======================================================================
# 8. ANALYZE ONE FAILURE
# ======================================================================

def analyze_failure(case, failure_type):

    filename = case["filename"]

    image_path = find_image(filename)

    if image_path is None:

        print(f"[WARNING] Image not found: {filename}")
        return None

    image = cv2.imread(str(image_path))

    if image is None:

        print(f"[WARNING] Could not read: {image_path}")
        return None

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    height, width = image.shape[:2]

    label_path = find_label(image_path)

    ground_truth = load_yolo_labels(
        label_path,
        width,
        height
    )

    can_boxes = [
        x for x in ground_truth
        if x["class_id"] == CAN_ID
    ]

    if len(can_boxes) == 0:
        print(f"[WARNING] No CAN label found: {filename}")
        return None

    # Run model
    results = model.predict(
        source=str(image_path),
        imgsz=640,
        conf=CONF_THRESHOLD,
        verbose=False
    )

    result = results[0]

    predictions = []

    if result.boxes is not None:

        for i in range(len(result.boxes)):

            cls = int(
                result.boxes.cls[i].item()
            )

            conf = float(
                result.boxes.conf[i].item()
            )

            box = result.boxes.xyxy[i].cpu().numpy().tolist()

            predictions.append({
                "class_id": cls,
                "class_name": CLASS_NAMES.get(cls, str(cls)),
                "confidence": conf,
                "xyxy": box
            })

    # Find best prediction for each CAN
    for can in can_boxes:

        best_prediction = None
        best_iou = 0.0

        for pred in predictions:

            iou = calculate_iou(
                can["xyxy"],
                pred["xyxy"]
            )

            if iou > best_iou:

                best_iou = iou
                best_prediction = pred

        can["best_prediction"] = best_prediction
        can["best_iou"] = best_iou

    # Select the relevant CAN
    can = can_boxes[0]

    pred = can["best_prediction"]

    if pred is not None:

        pred_class = pred["class_name"]
        confidence = pred["confidence"]

    else:

        pred_class = "NO DETECTION"
        confidence = 0.0

    # Create visualization
    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.imshow(image_rgb)

    # Ground truth CAN
    gx1, gy1, gx2, gy2 = can["xyxy"]

    rect_gt = plt.Rectangle(
        (gx1, gy1),
        gx2 - gx1,
        gy2 - gy1,
        fill=False,
        linewidth=3,
        linestyle="--"
    )

    ax.add_patch(rect_gt)

    ax.text(
        gx1,
        max(gy1 - 8, 5),
        "GROUND TRUTH: CAN",
        fontsize=11,
        fontweight="bold"
    )

    # Prediction
    if pred is not None:

        px1, py1, px2, py2 = pred["xyxy"]

        rect_pred = plt.Rectangle(
            (px1, py1),
            px2 - px1,
            py2 - py1,
            fill=False,
            linewidth=3
        )

        ax.add_patch(rect_pred)

        ax.text(
            px1,
            min(py2 + 18, height - 5),
            f"PRED: {pred_class} "
            f"conf={confidence:.3f} "
            f"IoU={can['best_iou']:.3f}",
            fontsize=11,
            fontweight="bold"
        )

    # Metadata
    area = can["area"]
    ar = can["aspect_ratio"]

    cx = can["xc"]
    cy = can["yc"]

    metadata = (
        f"Failure: {failure_type.upper()}\n"
        f"Image: {image_path.name}\n"
        f"CAN area: {area:.4f}\n"
        f"Aspect ratio: {ar:.3f}\n"
        f"Center: ({cx:.3f}, {cy:.3f})\n"
        f"Prediction: {pred_class}\n"
        f"Confidence: {confidence:.3f}\n"
        f"IoU: {can['best_iou']:.3f}"
    )

    ax.text(
        0.01,
        0.99,
        metadata,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round",
            facecolor="white",
            alpha=0.85
        )
    )

    ax.axis("off")

    plt.tight_layout()

    # Safe filename
    safe_name = image_path.stem.replace(
        " ",
        "_"
    )

    if failure_type == "MISSED":

        output_path = MISSED_DIR / f"{safe_name}_MISSED.jpg"

    else:

        output_path = WRONG_DIR / f"{safe_name}_WRONG_CLASS.jpg"

    plt.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight"
    )

    plt.close(fig)

    return {
        "filename": image_path.name,
        "failure_type": failure_type,
        "ground_truth_class": "CAN",
        "predicted_class": pred_class,
        "confidence": confidence,
        "iou": can["best_iou"],
        "area": can["area"],
        "aspect_ratio": can["aspect_ratio"],
        "center_x": can["xc"],
        "center_y": can["yc"],
        "output_image": str(output_path)
    }


# ======================================================================
# 9. PROCESS MISSED CAN CASES
# ======================================================================

print("\n" + "=" * 70)
print("PROCESSING MISSED CAN CASES")
print("=" * 70)

visual_results = []

for i, case in enumerate(
    missed_cases,
    start=1
):

    print(
        f"[MISSED {i}/{len(missed_cases)}] "
        f"{case['filename']}"
    )

    result = analyze_failure(
        case,
        "MISSED"
    )

    if result is not None:
        visual_results.append(result)


# ======================================================================
# 10. PROCESS WRONG-CLASS CASES
# ======================================================================

print("\n" + "=" * 70)
print("PROCESSING WRONG-CLASS CAN CASES")
print("=" * 70)

for i, case in enumerate(
    wrong_cases,
    start=1
):

    print(
        f"[WRONG {i}/{len(wrong_cases)}] "
        f"{case['filename']}"
    )

    result = analyze_failure(
        case,
        "WRONG_CLASS"
    )

    if result is not None:
        visual_results.append(result)


# ======================================================================
# 11. SAVE VISUAL ANALYSIS JSON
# ======================================================================

json_output = OUTPUT_ROOT / "section5_visual_failure_analysis.json"

with open(
    json_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        visual_results,
        f,
        indent=4
    )


# ======================================================================
# 12. FINAL SUMMARY
# ======================================================================

print("\n" + "=" * 70)
print("SECTION 5 — VISUAL INSPECTION COMPLETED")
print("=" * 70)

print(f"Missed CAN cases      : {len(missed_cases)}")
print(f"Wrong-class CAN cases : {len(wrong_cases)}")
print(f"Visualizations saved  : {len(visual_results)}")

print("\nOutput directories:")
print(f"Missed CAN      : {MISSED_DIR}")
print(f"Wrong-class CAN : {WRONG_DIR}")

print("\nAnalysis JSON:")
print(json_output)

print("\n" + "=" * 70)
print("[OK] SECTION 5 COMPLETED")
print("=" * 70)

EXPERIMENT 2 — CAN FAILURE ANALYSIS
SECTION 5 — VISUAL CAN FAILURE INSPECTION

[1] Configuration
----------------------------------------------------------------------
Model       : G:\EcoBotX_YOLO_training\experiment2_ecobotx_light-2\weights\best.pt
Test images : G:\EcoBotX_YOLO\images\test
Test labels : G:\EcoBotX_YOLO\labels\test
Section 4   : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\can_failure_characterization.json
Output      : G:\EcoBotX_YOLO_training\experiment2_can_failure_analysis\section5_visual_inspection

[2] Checking required files...
[OK] Model found
[OK] Test image directory found
[OK] Test label directory found
[OK] Section 4 characterization found

[3] Loading Experiment 2 model...
[OK] Model loaded
Model classes: {0: 'BOTTLE', 1: 'CAN', 2: 'PAPER', 3: 'WRAPPER'}

[4] Loading Section 4 results...
[OK] Section 4 results loaded

[5] Extracting failure cases...
[INFO] Missed cases found      : 0
[INFO] Wrong-class cases found : 0

[INFO] Searching missed